# Notebook 05 — Reproduction of an Established PhysioNet 2019 LSTM Pipeline

This notebook reproduces and adapts the workflow from **nerajbobra/sepsis-prediction**, an open-source PhysioNet Computing in Cardiology Challenge 2019 project.

The original project:
- uses 10 hours of ICU history to predict sepsis in the following hour;
- keeps HR, MAP, O2Sat, SBP and Resp as a 10-step time series;
- summarizes the remaining variables by the median over the 10-hour window;
- standardizes continuous variables using training-patient statistics;
- uses a two-branch neural network with Bidirectional LSTMs + dense layers;
- handles class imbalance with class weights;
- reports a test AUC of about 0.76 on its setup.

This notebook is an **adaptation for our Training Set A only**. Therefore, its final metrics should not be expected to exactly reproduce the repository's 0.76 AUC, because the original work used a larger patient population and its own train/test split.

Reference:
https://github.com/nerajbobra/sepsis-prediction

## Important methodological note

This notebook intentionally does **not** use the symbolic PrefixSpan / distance-feature pipeline from Notebooks 01–05.

The purpose here is to obtain a strong, reproducible baseline from an established implementation on the same PhysioNet 2019 dataset. We can later compare this established LSTM approach against the sequential-pattern approach required by the project proposal.

The original repository uses the challenge's `SepsisLabel` differently: it shifts the positive prediction target so that the model predicts sepsis in the next hour rather than reproducing the challenge's six-hour-ahead label definition.

In [ ]:
# Core imports
import os
import glob
import pickle
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve
)

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 1. Configuration

Set `DATA_DIR` to the folder containing the PhysioNet Training Set A `.psv` files.

Expected structure can be either:

`DATA_DIR/*.psv`

or a nested directory containing the `.psv` files.

The default configuration uses all available Training Set A patients. For a quick smoke test, set `MAX_PATIENTS` to a small number such as 300.

In [ ]:
# CHANGE THIS PATH
DATA_DIR = Path("./data/training")

# Output directory for this reproduction
OUTPUT_DIR = Path("./outputs/reproduction_nerajbobra")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# None = use all available patients
MAX_PATIENTS = None

WINDOW_LEN = 10
PRED_LEN = 1
BATCH_SIZE = 64
EPOCHS = 50

print("DATA_DIR:", DATA_DIR.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

In [ ]:
# Discover PSV files recursively
psv_files = sorted(DATA_DIR.rglob("*.psv"))

if MAX_PATIENTS is not None:
    psv_files = psv_files[:MAX_PATIENTS]

print(f"PSV files found: {len(psv_files):,}")

if len(psv_files) == 0:
    raise FileNotFoundError(
        f"No .psv files found under {DATA_DIR.resolve()}. "
        "Update DATA_DIR to your PhysioNet Training Set A folder."
    )

print("Example:", psv_files[0])

## 2. Load the PhysioNet PSV files

Each patient is represented by one PSV file with hourly measurements. We retain a patient identifier derived from the filename.

For this reproduction, the raw files are loaded directly rather than depending on the earlier notebooks.

In [ ]:
def load_psv_file(path):
    df = pd.read_csv(path, sep="|")
    df["patient"] = path.stem
    return df

sample_df = load_psv_file(psv_files[0])

print("Sample patient:", sample_df["patient"].iloc[0])
print("Shape:", sample_df.shape)
display(sample_df.head())

In [ ]:
# Combine the selected patient files.
# This can take a while because the PhysioNet files are large.

frames = []
for i, path in enumerate(psv_files, start=1):
    frames.append(load_psv_file(path))
    if i % 500 == 0 or i == len(psv_files):
        print(f"Loaded {i:,}/{len(psv_files):,} patients")

df = pd.concat(frames, ignore_index=True)

print("\nCombined shape:", df.shape)
print("Patients:", df["patient"].nunique())
print("Sepsis-labelled rows:", int(df["SepsisLabel"].sum()))

## 3. Reproduce the original feature grouping

The reference implementation separates variables into two groups.

Time-series branch:
- HR
- MAP
- O2Sat
- SBP
- Resp

Window-summary branch:
- Unit1
- Gender
- HospAdmTime
- Age
- DBP
- Temp
- Glucose
- Potassium
- Hct
- FiO2
- Hgb
- pH
- BUN
- WBC
- Magnesium
- Creatinine
- Platelets
- Calcium
- PaCO2
- BaseExcess
- Chloride
- HCO3
- Phosphate
- EtCO2
- SaO2
- PTT
- Lactate
- AST
- Alkalinephos
- Bilirubin_total
- TroponinI
- Fibrinogen
- Bilirubin_direct

The reference drops Unit2 and ICULOS.

In [ ]:
cols_to_drop = ["Unit2", "ICULOS"]

cols_cont = ["HR", "MAP", "O2Sat", "SBP", "Resp"]

cols_to_bin = [
    "Unit1", "Gender", "HospAdmTime", "Age", "DBP", "Temp", "Glucose",
    "Potassium", "Hct", "FiO2", "Hgb", "pH", "BUN", "WBC", "Magnesium",
    "Creatinine", "Platelets", "Calcium", "PaCO2", "BaseExcess", "Chloride",
    "HCO3", "Phosphate", "EtCO2", "SaO2", "PTT", "Lactate", "AST",
    "Alkalinephos", "Bilirubin_total", "TroponinI", "Fibrinogen",
    "Bilirubin_direct"
]

required_cols = set(cols_cont + cols_to_bin + ["SepsisLabel", "patient"])

missing_cols = sorted(required_cols - set(df.columns))
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print("Continuous sequence variables:", cols_cont)
print("Window-summary variables:", len(cols_to_bin))

In [ ]:
missing_pct = df.isna().mean().sort_values(ascending=False)

display(
    missing_pct.to_frame("missing_fraction")
    .assign(missing_percent=lambda x: 100 * x["missing_fraction"])
    .head(20)
)

print("\nSepsis patients:", df.groupby("patient")["SepsisLabel"].max().sum())
print("Non-sepsis patients:", df["patient"].nunique() - df.groupby("patient")["SepsisLabel"].max().sum())

## 4. Patient-level train/test split

The reference implementation reserves 6,000 patients as a held-out test population and computes standardization statistics using the remaining patients.

Because we are using Training Set A only, the notebook applies the same idea proportionally:
- 70% of patients → development/training population
- 30% of patients → held-out test population

The split is performed at patient level, so windows from the same patient cannot appear in both sets.

In [ ]:
patients = np.array(sorted(df["patient"].unique()))
rng = np.random.default_rng(SEED)
rng.shuffle(patients)

n_test = max(1, int(round(0.30 * len(patients))))

test_patients = set(patients[:n_test])
train_patients = set(patients[n_test:])

print("Training patients:", len(train_patients))
print("Test patients:", len(test_patients))

assert train_patients.isdisjoint(test_patients)

## 5. Training-set standardization statistics

The reference implementation computes means and standard deviations from training patients only, then applies those values to both training and test data.

We reproduce that principle here.

In [ ]:
train_rows = df[df["patient"].isin(train_patients)]

scaling_cols = cols_cont + [
    c for c in cols_to_bin if c not in ["Gender", "Unit1"]
]

mean_std = train_rows[scaling_cols].agg(["mean", "std"]).T

# Avoid division by zero for any constant column.
mean_std["std"] = mean_std["std"].replace(0, 1.0)

mean_std.to_csv(OUTPUT_DIR / "mean_std_scaling.csv")

print("Scaling statistics calculated from training patients only.")
display(mean_std.head(10))

## 6. Recreate the 10-hour → next-hour prediction windows

For every patient:

1. Missing values in HR/MAP/O2Sat/SBP/Resp are backfilled and then forward-filled.
2. These five variables are standardized and retained as a 10 × 5 sequence.
3. Every other variable is summarized by its median over the 10-hour window.
4. The label is the `SepsisLabel` value in the following hour.
5. The window slides forward by one hour.

Windows where one of the five sequence variables remains NaN after filling are skipped, matching the reference implementation's logic.

In [ ]:
def standardize(series, col):
    return (series - mean_std.loc[col, "mean"]) / mean_std.loc[col, "std"]

def make_windows_for_patient(group):
    group = group.sort_index().reset_index(drop=True).copy()

    # Fill the five relatively dense time-series variables.
    for col in cols_cont:
        group[col] = group[col].bfill().ffill()

    # Standardize the five time-series variables.
    for col in cols_cont:
        group[col] = standardize(group[col], col)

    windows_cont = []
    windows_summary = []
    labels = []
    patient_ids = []

    n = len(group)

    for i in range(0, n - WINDOW_LEN - PRED_LEN + 1):
        tmp = group.iloc[i:i + WINDOW_LEN]
        target = group.iloc[i + WINDOW_LEN:i + WINDOW_LEN + PRED_LEN]

        X_cont = tmp[cols_cont].to_numpy(dtype=np.float32)

        if np.isnan(X_cont).any():
            continue

        summary = {}
        for col in cols_to_bin:
            value = tmp[col].median()

            if col not in ["Gender", "Unit1"]:
                value = (value - mean_std.loc[col, "mean"]) / mean_std.loc[col, "std"]

            summary[col] = value

        windows_cont.append(X_cont)
        windows_summary.append([summary[c] for c in cols_to_bin])
        labels.append(int(target["SepsisLabel"].any()))
        patient_ids.append(group["patient"].iloc[0])

    return windows_cont, windows_summary, labels, patient_ids

In [ ]:
def build_window_dataset(patient_set, label):
    subset = df[df["patient"].isin(patient_set)]

    all_cont = []
    all_summary = []
    all_labels = []
    all_patients = []

    grouped = subset.groupby("patient", sort=False)

    for idx, (patient, group) in enumerate(grouped, start=1):
        cont, summary, labels, ids = make_windows_for_patient(group)

        all_cont.extend(cont)
        all_summary.extend(summary)
        all_labels.extend(labels)
        all_patients.extend(ids)

        if idx % 500 == 0 or idx == len(grouped):
            print(f"{label}: processed {idx:,}/{len(grouped):,} patients")

    X_cont = np.asarray(all_cont, dtype=np.float32)
    X_summary = np.asarray(all_summary, dtype=np.float32)
    y = np.asarray(all_labels, dtype=np.int32)
    patient_ids = np.asarray(all_patients)

    return X_cont, X_summary, y, patient_ids

In [ ]:
# Build the development and held-out test windows.
# This is the main preprocessing step and may take some time.

X_dev_cont, X_dev_summary, y_dev, dev_patient_ids = build_window_dataset(
    train_patients, "TRAIN"
)

X_test_cont, X_test_summary, y_test, test_patient_ids = build_window_dataset(
    test_patients, "TEST"
)

print("\nTRAIN/DEV WINDOWS")
print("X_cont:", X_dev_cont.shape)
print("X_summary:", X_dev_summary.shape)
print("y:", y_dev.shape)
print("Positive:", int(y_dev.sum()))

print("\nTEST WINDOWS")
print("X_cont:", X_test_cont.shape)
print("X_summary:", X_test_summary.shape)
print("y:", y_test.shape)
print("Positive:", int(y_test.sum()))

In [ ]:
# Save the windowed arrays so rerunning the notebook does not require preprocessing again.
np.save(OUTPUT_DIR / "X_dev_cont.npy", X_dev_cont)
np.save(OUTPUT_DIR / "X_dev_summary.npy", X_dev_summary)
np.save(OUTPUT_DIR / "y_dev.npy", y_dev)
np.save(OUTPUT_DIR / "dev_patient_ids.npy", dev_patient_ids)

np.save(OUTPUT_DIR / "X_test_cont.npy", X_test_cont)
np.save(OUTPUT_DIR / "X_test_summary.npy", X_test_summary)
np.save(OUTPUT_DIR / "y_test.npy", y_test)
np.save(OUTPUT_DIR / "test_patient_ids.npy", test_patient_ids)

print("Saved preprocessed arrays.")

## 7. Train/validation split

The reference project randomly splits its development windows into training and validation sets.

For consistency with that implementation, we use an 80/20 window-level split here. The held-out test patients remain untouched until final evaluation.

In [ ]:
from sklearn.model_selection import train_test_split

(
    X_train_cont,
    X_val_cont,
    X_train_summary,
    X_val_summary,
    y_train,
    y_val
) = train_test_split(
    X_dev_cont,
    X_dev_summary,
    y_dev,
    test_size=0.20,
    random_state=SEED,
    stratify=y_dev
)

print("Train:", X_train_cont.shape, y_train.shape)
print("Validation:", X_val_cont.shape, y_val.shape)
print("Test:", X_test_cont.shape, y_test.shape)

print("\nPositive rates:")
print("Train:", y_train.mean())
print("Val:  ", y_val.mean())
print("Test: ", y_test.mean())

## 8. Handle NaN values in the summary branch

The reference model uses a Keras `Masking(mask_value=np.pi)` layer.

Therefore, remaining NaN values in the window-summary branch are replaced with `np.pi`. The masking layer can then ignore those values.

In [ ]:
X_train_summary_masked = np.where(np.isnan(X_train_summary), np.pi, X_train_summary).astype(np.float32)
X_val_summary_masked = np.where(np.isnan(X_val_summary), np.pi, X_val_summary).astype(np.float32)
X_test_summary_masked = np.where(np.isnan(X_test_summary), np.pi, X_test_summary).astype(np.float32)

print("NaNs after masking:")
print("Train:", np.isnan(X_train_summary_masked).sum())
print("Val:  ", np.isnan(X_val_summary_masked).sum())
print("Test: ", np.isnan(X_test_summary_masked).sum())

## 9. Class weights

Sepsis examples are highly imbalanced. The reference implementation compensates using class-weighted categorical cross-entropy rather than oversampling the positive class.

In [ ]:
count_class_0 = np.sum(y_train == 0)
count_class_1 = np.sum(y_train == 1)

max_count = max(count_class_0, count_class_1)

class_weights = {
    0: max_count / count_class_0,
    1: max_count / count_class_1
}

print("Class counts:")
print("No sepsis:", count_class_0)
print("Sepsis:   ", count_class_1)
print("\nClass weights:", class_weights)

## 10. Define the reproduced two-branch Bidirectional LSTM model

This follows the architecture reported in the reference repository:

Time-series branch:
- BiLSTM(100)
- BiLSTM(75)
- Dense(35)
- BatchNorm
- Dense(15)
- BatchNorm

Summary branch:
- Masking
- Dense(30)
- BatchNorm
- Dense(15)
- BatchNorm

The two 15-dimensional representations are added and passed to a 2-class softmax output.

In [ ]:
import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (
    LSTM, Dense, Bidirectional, BatchNormalization,
    Masking, Add
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

tf.random.set_seed(SEED)

INPUT_SEQ_LEN = WINDOW_LEN
INPUT_NUM_CH = len(cols_cont)
INPUT_SUMMARY_FEATS = len(cols_to_bin)

input1 = Input(shape=(INPUT_SEQ_LEN, INPUT_NUM_CH), name="timeseries_input")

model1 = Bidirectional(
    LSTM(100, kernel_regularizer=l2(0.001), return_sequences=True)
)(input1)

model1 = Bidirectional(
    LSTM(75, kernel_regularizer=l2(0.001))
)(model1)

model1 = Dense(
    35, kernel_regularizer=l2(0.001), activation="relu"
)(model1)

model1 = BatchNormalization()(model1)

model1 = Dense(
    15, kernel_regularizer=l2(0.001), activation="relu"
)(model1)

model1 = BatchNormalization()(model1)

input2 = Input(
    shape=(INPUT_SUMMARY_FEATS,),
    name="summary_input"
)

model2 = Masking(mask_value=np.pi)(input2)

model2 = Dense(
    30, kernel_regularizer=l2(0.001), activation="relu"
)(model2)

model2 = BatchNormalization()(model2)

model2 = Dense(
    15, kernel_regularizer=l2(0.001), activation="relu"
)(model2)

model2 = BatchNormalization()(model2)

merged = Add()([model1, model2])

output = Dense(
    2, kernel_regularizer=l2(0.001), activation="softmax",
    name="output"
)(merged)

model = Model(inputs=[input1, input2], outputs=output)

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

## 11. Train

The reference repository trains for up to 50 epochs with:
- batch size 64;
- early stopping on validation loss;
- best-model checkpointing;
- class weights.

We keep those settings as the default reproduction configuration.

In [ ]:
from tensorflow.keras.utils import to_categorical

y_train_cat = to_categorical(y_train, num_classes=2)
y_val_cat = to_categorical(y_val, num_classes=2)
y_test_cat = to_categorical(y_test, num_classes=2)

model_path = OUTPUT_DIR / "nerajbobra_reproduction_best.keras"

checkpoint = ModelCheckpoint(
    model_path,
    monitor="val_loss",
    verbose=1,
    save_best_only=True,
    mode="min"
)

earlystop = EarlyStopping(
    monitor="val_loss",
    min_delta=0,
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    [X_train_cont, X_train_summary_masked],
    y_train_cat,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(
        [X_val_cont, X_val_summary_masked],
        y_val_cat
    ),
    callbacks=[earlystop, checkpoint],
    class_weight=class_weights,
    verbose=1
)

## 12. Training curves

In [ ]:
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(8, 5))
plt.plot(history_df["loss"], label="Train loss")
plt.plot(history_df["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_validation_loss.png", dpi=200)
plt.show()

## 13. Final evaluation

We report:
- AUROC
- AUPRC
- accuracy
- precision
- recall
- F1

AUROC and especially AUPRC are important here because the sepsis class is strongly imbalanced.

In [ ]:
# Ensure the best checkpoint is loaded.
model = tf.keras.models.load_model(model_path)

val_prob = model.predict(
    [X_val_cont, X_val_summary_masked],
    batch_size=BATCH_SIZE,
    verbose=0
)[:, 1]

test_prob = model.predict(
    [X_test_cont, X_test_summary_masked],
    batch_size=BATCH_SIZE,
    verbose=0
)[:, 1]

val_pred = (val_prob >= 0.5).astype(int)
test_pred = (test_prob >= 0.5).astype(int)

def evaluate_binary(y_true, prob, pred, split_name):
    return {
        "split": split_name,
        "AUROC": roc_auc_score(y_true, prob),
        "AUPRC": average_precision_score(y_true, prob),
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
    }

results = pd.DataFrame([
    evaluate_binary(y_val, val_prob, val_pred, "Validation"),
    evaluate_binary(y_test, test_prob, test_pred, "Test")
])

display(results)

results.to_csv(OUTPUT_DIR / "reproduction_metrics.csv", index=False)

## 14. ROC and Precision–Recall curves

In [ ]:
fpr, tpr, _ = roc_curve(y_test, test_prob)
precision, recall, _ = precision_recall_curve(y_test, test_prob)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, label=f"AUROC = {roc_auc_score(y_test, test_prob):.3f}")
ax.plot([0, 1], [0, 1], "--")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Test ROC Curve")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "test_roc.png", dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(
    recall,
    precision,
    label=f"AUPRC = {average_precision_score(y_test, test_prob):.3f}"
)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Test Precision–Recall Curve")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "test_pr.png", dpi=200)
plt.show()

## 15. Confusion matrix and classification report

In [ ]:
cm = confusion_matrix(y_test, test_pred)

print("Confusion matrix:")
print(cm)

print("\nClassification report:")
print(
    classification_report(
        y_test,
        test_pred,
        target_names=["No Sepsis", "Sepsis"],
        zero_division=0
    )
)

## 16. Compare with the reference result

The reference repository reports a test **AUC ≈ 0.76** on its own experimental setup.

Our result is an adaptation because we are using Training Set A only and a different patient-level split. Therefore:

- do not claim that our result reproduces 0.76 exactly unless it actually does;
- report our measured value as the result of the adapted reproduction;
- cite the original repository for the 0.76 reference result.

A successful run should demonstrate that the established LSTM approach provides a substantially stronger temporal baseline than the earlier five-pattern distance-feature model.

In [ ]:
reference_auc = 0.76
test_auc = float(results.loc[results["split"] == "Test", "AUROC"].iloc[0])

print(f"Reference repository AUC: ~{reference_auc:.2f}")
print(f"Our Training Set A test AUROC: {test_auc:.4f}")
print(f"Absolute difference: {test_auc - reference_auc:+.4f}")

## 17. Save a compact experiment summary

This file can later be used directly in the project report when comparing the reproduced LSTM baseline with the sequential-pattern approach.

In [ ]:
summary = {
    "method": "Neraj Bobra-style two-branch Bidirectional LSTM",
    "dataset": "PhysioNet/CinC 2019 Training Set A",
    "window_hours": WINDOW_LEN,
    "prediction_horizon_hours": PRED_LEN,
    "sequence_features": ", ".join(cols_cont),
    "summary_feature_count": len(cols_to_bin),
    "batch_size": BATCH_SIZE,
    "max_epochs": EPOCHS,
    "test_AUROC": test_auc,
    "test_AUPRC": float(results.loc[results["split"] == "Test", "AUPRC"].iloc[0]),
}

with open(OUTPUT_DIR / "experiment_summary.txt", "w") as f:
    for k, v in summary.items():
        f.write(f"{k}: {v}\n")

print("Saved:", OUTPUT_DIR / "experiment_summary.txt")

## Reproduction conclusion

This notebook establishes an independent, literature/repository-backed temporal deep-learning baseline on the same PhysioNet 2019 dataset.

The key experimental distinction from the original project is the dataset restriction to Training Set A. The original repository used a larger population and reported AUC ≈ 0.76.

For the final project, this baseline can be presented alongside the proposed sequential-pattern pipeline rather than replacing the project's sequential-mining objective.